In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss

In [51]:
pd.set_option('display.max_columns', None)
train = pd.read_csv('cardihack_final_train.csv')
test = pd.read_csv('cardihack_final_test.csv')

In [52]:
snp_cols = [c for c in train.columns if c.startswith('SNP')]
severity_weights = {0: 1.5, 1: 1.0}
mace_weights = {0: 1, 1: 3, 2: 4}

# Helper Functions

In [53]:
def _weighted_mean_abs_shap(shap_vals: np.ndarray, weights: np.ndarray | None) -> np.ndarray:
    """
    Accepts:
      - 2D: (n_samples, n_features)
      - 3D: (n_samples, n_classes, n_features)  [multiclass]
    Returns:
      - (n_features,)
    """
    abs_vals = np.abs(shap_vals)

    if weights is None:
        # unweighted mean over samples (and classes if present)
        if abs_vals.ndim == 2:
            return abs_vals.mean(axis=0)
        elif abs_vals.ndim == 3:
            return abs_vals.mean(axis=(0, 1))
        else:
            raise ValueError(f"Unexpected shap_vals ndim={abs_vals.ndim}")

    w = np.asarray(weights, dtype=float).reshape(-1)
    denom = float(np.sum(w))
    if denom <= 0:
        if abs_vals.ndim == 2:
            return abs_vals.mean(axis=0)
        elif abs_vals.ndim == 3:
            return abs_vals.mean(axis=(0, 1))
        else:
            raise ValueError(f"Unexpected shap_vals ndim={abs_vals.ndim}")

    if abs_vals.ndim == 2:
        w2 = w.reshape(-1, 1)
        return (abs_vals * w2).sum(axis=0) / denom

    if abs_vals.ndim == 3:
    
        w3 = w.reshape(-1, 1, 1)
        weighted = (abs_vals * w3).sum(axis=0) / denom 
        return weighted.mean(axis=0)  

    raise ValueError(f"Unexpected shap_vals ndim={abs_vals.ndim}")

In [54]:
# SEVERITY set up
def weighted_logloss_binary(y_true: np.ndarray, p1: np.ndarray, label_weights: dict[int, float]) -> float:
    # manual weighted logloss to avoid sklearn dependency here
    eps = 1e-15
    p1 = np.clip(np.asarray(p1, dtype=float), eps, 1 - eps)
    y_true = np.asarray(y_true, dtype=int)
    w = np.array([label_weights[int(y)] for y in y_true], dtype=float)

    ll = -(y_true * np.log(p1) + (1 - y_true) * np.log(1 - p1))
    return float(np.sum(w * ll) / np.sum(w))


def weighted_dummy_logloss_binary(y_true: np.ndarray, label_weights: dict[int, float]) -> float:
    p = float(np.mean(np.asarray(y_true, dtype=int)))
    p1 = np.full(len(y_true), p, dtype=float)
    return weighted_logloss_binary(y_true, p1, label_weights)


def weighted_worst_logloss_binary(y_true: np.ndarray, label_weights: dict[int, float], eps: float = 1e-15) -> float:
    y_true = np.asarray(y_true, dtype=int)
    # worst predicts 1 when y=0, and 0 when y=1
    p1 = np.where(y_true == 1, eps, 1.0 - eps).astype(float)
    return weighted_logloss_binary(y_true, p1, label_weights)


def rescaled_weighted_logloss(weighted_ll: float, dummy_ll: float, worst_ll: float) -> float:
    if dummy_ll <= 0:
        return 0.0
    if weighted_ll <= dummy_ll:
        return float(1.0 - (weighted_ll / dummy_ll))
    denom = (worst_ll - dummy_ll)
    if denom <= 0:
        return float(-(weighted_ll - dummy_ll))
    return float(-(weighted_ll - dummy_ll) / denom)

In [55]:
# MACE: weighted QWK

def _qwk_weights_matrix(n_classes: int = 3) -> np.ndarray:
    W = np.zeros((n_classes, n_classes), dtype=float)
    for i in range(n_classes):
        for j in range(n_classes):
            W[i, j] = ((i - j) ** 2) / ((n_classes - 1) ** 2)
    return W


def quadratic_weighted_kappa(y_true: np.ndarray, y_pred: np.ndarray, sample_weight: np.ndarray | None = None) -> float:
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    assert y_true.shape == y_pred.shape

    n_classes = 3
    if sample_weight is None:
        sample_weight = np.ones_like(y_true, dtype=float)
    else:
        sample_weight = np.asarray(sample_weight, dtype=float)

    W = _qwk_weights_matrix(n_classes)

    O = np.zeros((n_classes, n_classes), dtype=float)
    for yt, yp, w in zip(y_true, y_pred, sample_weight):
        O[yt, yp] += w

    hist_true = np.zeros(n_classes, dtype=float)
    hist_pred = np.zeros(n_classes, dtype=float)
    for yt, yp, w in zip(y_true, y_pred, sample_weight):
        hist_true[yt] += w
        hist_pred[yp] += w

    total = float(np.sum(sample_weight))
    if total <= 0:
        return 0.0

    E = np.outer(hist_true, hist_pred) / total
    num = float((W * O).sum())
    den = float((W * E).sum())
    if den == 0:
        return 0.0
    return float(1.0 - num / den)

def apply_thresholds_ordinal(score: np.ndarray, t1: float, t2: float) -> np.ndarray:
    """
    Map a 1D ordinal score into classes 0/1/2 using thresholds.
    """
    score = np.asarray(score, dtype=float)
    yhat = np.zeros_like(score, dtype=int)
    yhat[score >= t1] = 1
    yhat[score >= t2] = 2
    return yhat

def optimize_thresholds_for_weighted_qwk(
    y_true: np.ndarray,
    score: np.ndarray,
    sample_weight: np.ndarray,
    n_grid: int = 40
) -> tuple[float, float, float]:
    """
    Brute-force-ish threshold search on quantile grid:
    returns (best_t1, best_t2, best_qwk)
    """
    y_true = np.asarray(y_true, dtype=int)
    score = np.asarray(score, dtype=float)
    sample_weight = np.asarray(sample_weight, dtype=float)

    qs = np.linspace(0.05, 0.95, n_grid)
    grid = np.quantile(score, qs)

    best = (-np.inf, None, None)  # (qwk, t1, t2)
    for i in range(len(grid)):
        for j in range(i + 1, len(grid)):
            t1, t2 = float(grid[i]), float(grid[j])
            yhat = apply_thresholds_ordinal(score, t1, t2)
            qwk = quadratic_weighted_kappa(y_true, yhat, sample_weight=sample_weight)
            if qwk > best[0]:
                best = (qwk, t1, t2)

    return best[1], best[2], float(best[0])

In [56]:
from dataclasses import dataclass
from typing import Callable, Iterable, Optional, Sequence, Tuple

@dataclass(frozen=True)
class CVFold:
    """A single CV fold split (train/valid indices into the *training split* arrays)."""
    train_idx: np.ndarray
    valid_idx: np.ndarray

def make_stratified_folds(
    y: np.ndarray,
    n_splits: int,
    seed: int,
) -> list[CVFold]:
    """Return StratifiedKFold splits as a list."""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return [CVFold(tr_i, va_i) for tr_i, va_i in skf.split(np.zeros(len(y)), y)]

def train_one_fold_lgbm(
    *,
    params: dict,
    X_tr: np.ndarray,
    y_tr: np.ndarray,
    X_va: np.ndarray,
    y_va: np.ndarray,
    feature_cols: Sequence[str],
    num_boost_round: int,
    early_stopping_rounds: int,
    verbose: bool,
    w_tr: Optional[np.ndarray] = None,
    w_va: Optional[np.ndarray] = None,
) -> tuple[lgb.Booster, int, np.ndarray]:
    """Train one LightGBM model on a fold and return (booster, best_it, pred_valid)."""
    dtrain = lgb.Dataset(
        X_tr, label=y_tr, weight=w_tr, feature_name=list(feature_cols), free_raw_data=True
    )
    dvalid = lgb.Dataset(
        X_va, label=y_va, weight=w_va, feature_name=list(feature_cols), reference=dtrain, free_raw_data=True
    )

    booster = lgb.train(
        params=params,
        train_set=dtrain,
        num_boost_round=num_boost_round,
        valid_sets=[dvalid],
        valid_names=["valid"],
        callbacks=[
            lgb.early_stopping(early_stopping_rounds, verbose=verbose),
            lgb.log_evaluation(period=0 if not verbose else 50),
        ],
    )
    best_it = int(booster.best_iteration or num_boost_round)
    pred_va = booster.predict(X_va, num_iteration=best_it)
    return booster, best_it, pred_va

def sample_params_binary(rng: np.random.Generator) -> dict:
    """Sample parameters set for binary classification."""
    max_depth = int(rng.integers(3, 11))
    max_leaves = max(8, min(2 ** max_depth, 256))
    num_leaves = int(rng.integers(8, max_leaves + 1))
    return {
        "objective": "binary",
        "metric": "binary_logloss",
        "boosting_type": "gbdt",
        "learning_rate": float(rng.uniform(0.01, 0.15)),
        "max_depth": max_depth,
        "num_leaves": num_leaves,
        "min_data_in_leaf": int(rng.integers(10, 120)),
        "min_sum_hessian_in_leaf": float(rng.uniform(1e-3, 5.0)),
        "feature_fraction": float(rng.uniform(0.6, 1.0)),
        "bagging_fraction": float(rng.uniform(0.6, 1.0)),
        "bagging_freq": int(rng.integers(1, 7)),
        "lambda_l1": float(rng.uniform(0.0, 10.0)),
        "lambda_l2": float(rng.uniform(0.0, 10.0)),
        "verbosity": -1,
    }

def sample_params_multiclass(rng: np.random.Generator, num_class: int = 3) -> dict:
    """Sample parameters set for multiclass classification."""
    max_depth = int(rng.integers(3, 11))
    max_leaves = max(8, min(2 ** max_depth, 256))
    num_leaves = int(rng.integers(8, max_leaves + 1))
    return {
        "objective": "multiclass",
        "metric": "multi_logloss",
        "num_class": int(num_class),
        "boosting_type": "gbdt",
        "learning_rate": float(rng.uniform(0.01, 0.15)),
        "max_depth": max_depth,
        "num_leaves": num_leaves,
        "min_data_in_leaf": int(rng.integers(10, 120)),
        "min_sum_hessian_in_leaf": float(rng.uniform(1e-3, 5.0)),
        "feature_fraction": float(rng.uniform(0.6, 1.0)),
        "bagging_fraction": float(rng.uniform(0.6, 1.0)),
        "bagging_freq": int(rng.integers(1, 7)),
        "lambda_l1": float(rng.uniform(0.0, 10.0)),
        "lambda_l2": float(rng.uniform(0.0, 10.0)),
        "verbosity": -1,
    }


In [57]:
def _fit_lgbm_for_shap_binary(
    *,
    X_tr: pd.DataFrame,
    y_tr: np.ndarray,
    X_te: pd.DataFrame,
    y_te: np.ndarray,
    feature_cols: list[str],
    label_weights: dict[int, float],
    use_weights_for_training: bool,
    split_seed: int,
    num_boost_round: int,
    early_stopping_rounds: int,
) -> lgb.Booster:
    """Train binary LightGBM with rescaled weighted logloss."""
    # Baselines for rescaling
    dummy_ll_tr = weighted_dummy_logloss_binary(y_tr, label_weights)
    worst_ll_tr = weighted_worst_logloss_binary(y_tr, label_weights)

    # Weights
    w_tr_eval = np.array([label_weights[int(y)] for y in y_tr], dtype=float)
    w_te_eval = np.array([label_weights[int(y)] for y in y_te], dtype=float)
    w_tr_fit = w_tr_eval if use_weights_for_training else None
    w_te_fit = w_te_eval if use_weights_for_training else None

    def feval_rescaled_wll(preds: np.ndarray, dataset: lgb.Dataset):
        """Metric used ONLY for early stopping / model selection."""
        y_true = dataset.get_label().astype(int)
        wll = weighted_logloss_binary(y_true, preds, label_weights)
        rll = rescaled_weighted_logloss(wll, dummy_ll_tr, worst_ll_tr)
        return ("rescaled_wll", float(rll), True)

    params = {
        "objective": "binary",
        "metric": "None",  # use custom only
        "boosting_type": "gbdt",
        "learning_rate": 0.05,
        "num_leaves": 63,
        "max_depth": -1,
        "min_data_in_leaf": 30,
        "feature_fraction": 0.9,
        "bagging_fraction": 0.9,
        "bagging_freq": 1,
        "lambda_l2": 1.0,
        "lambda_l1": 0.0,
        "verbosity": -1,
        "seed": split_seed,
        "feature_pre_filter": False,
        "force_row_wise": True,
    }

    dtrain = lgb.Dataset(X_tr.values, label=y_tr, weight=w_tr_fit, feature_name=feature_cols, free_raw_data=True)
    dvalid = lgb.Dataset(X_te.values, label=y_te, weight=w_te_fit, feature_name=feature_cols, reference=dtrain, free_raw_data=True)

    booster = lgb.train(
        params=params,
        train_set=dtrain,
        num_boost_round=num_boost_round,
        valid_sets=[dvalid],
        valid_names=["valid"],
        feval=feval_rescaled_wll,
        callbacks=[
            lgb.early_stopping(early_stopping_rounds, verbose=False),
            lgb.log_evaluation(period=0),
        ],
    )
    return booster

def _compute_shap_importance(
    booster: lgb.Booster,
    X: pd.DataFrame,
    feature_cols: list[str],
    *,
    use_weights_for_shap: bool,
    sample_weight: np.ndarray | None = None,
) -> pd.Series:
    """
    Binary and Multiclass
    """
    n_feat = len(feature_cols)

    contrib = booster.predict(X.values, pred_contrib=True)

    # binary
    if contrib.ndim == 2 and contrib.shape[1] == n_feat + 1:
        shap_vals = contrib[:, :-1]  # drop expected value column
        if use_weights_for_shap and sample_weight is not None:
            scores = _weighted_mean_abs_shap(shap_vals, sample_weight)
        else:
            scores = np.mean(np.abs(shap_vals), axis=0)
        return pd.Series(scores, index=feature_cols)

    # multiclass
    if contrib.ndim != 2 or contrib.shape[1] % (n_feat + 1) != 0:
        raise ValueError(
            f"Unexpected pred_contrib shape {contrib.shape} for n_feat={n_feat}. "
            "Expected (n, p+1) or (n, K*(p+1))."
        )

    n_classes = contrib.shape[1] // (n_feat + 1)
    contrib_3d = contrib.reshape(contrib.shape[0], n_classes, n_feat + 1)
    shap_3d = contrib_3d[:, :, :-1]

    if use_weights_for_shap and sample_weight is not None:
        scores = _weighted_mean_abs_shap(shap_3d, sample_weight) 
    else:
        scores = np.mean(np.abs(shap_3d), axis=(0, 1))  # mean over samples AND classes

    return pd.Series(scores, index=feature_cols)

def rank_snps_by_shap_severity_lgbm(
    train: pd.DataFrame,
    target_col: str,
    base_cols: list[str],
    snp_cols: list[str],
    severity_label_weights: dict[int, float] = {0: 1.5, 1: 1.0},
    use_weights_for_training: bool = True,
    use_weights_for_shap: bool = True,
    test_size: float = 0.2,
    split_seed: int = 42,
    num_boost_round: int = 4000,
    early_stopping_rounds: int = 120,
) -> pd.DataFrame:
    
    snp_cols = [c for c in snp_cols if c in train.columns]
    if not snp_cols:
        raise ValueError("No SNP columns found in train.")

    feature_cols = base_cols + snp_cols
    y_all = train[target_col].astype(int)

    tr_idx, te_idx = train_test_split(
        train.index, test_size=test_size, stratify=y_all, random_state=split_seed
    )

    X_tr = train.loc[tr_idx, feature_cols].copy()
    X_te = train.loc[te_idx, feature_cols].copy()
    y_tr = y_all.loc[tr_idx].values
    y_te = y_all.loc[te_idx].values

    booster = _fit_lgbm_for_shap_binary(
        X_tr=X_tr,
        y_tr=y_tr,
        X_te=X_te,
        y_te=y_te,
        feature_cols=feature_cols,
        label_weights=severity_label_weights,
        use_weights_for_training=use_weights_for_training,
        split_seed=split_seed,
        num_boost_round=num_boost_round,
        early_stopping_rounds=early_stopping_rounds,
    )
    # Compute SHAP on the TRAIN split
    w_tr_eval = np.array([severity_label_weights[int(y)] for y in y_tr], dtype=float)
    shap_scores = _compute_shap_importance(
        booster,
        X_tr,
        feature_cols,
        use_weights_for_shap=use_weights_for_shap,
        sample_weight=w_tr_eval,
    )
    snp_scores = shap_scores.loc[snp_cols]
    rank_df = (
        snp_scores
        .rename("mean_abs_shap")
        .sort_values(ascending=False)
        .reset_index()
        .rename(columns={"index": "snp"})
    )
    rank_df["rank"] = np.arange(1, len(rank_df) + 1)

    return rank_df


In [58]:
def _fit_lgbm_for_shap_multiclass(
    *,
    X_tr: pd.DataFrame,
    y_tr: np.ndarray,
    X_te: pd.DataFrame,
    y_te: np.ndarray,
    feature_cols: list[str],
    label_weights: dict[int, float],
    use_weights_for_training: bool,
    split_seed: int,
    num_boost_round: int,
    early_stopping_rounds: int,
) -> lgb.Booster:
    """Train a multiclass LightGBM (0/1/2) with weighted QWK (thresholded)."""
    w_tr_eval = np.array([label_weights[int(y)] for y in y_tr], dtype=float)
    w_te_eval = np.array([label_weights[int(y)] for y in y_te], dtype=float)
    w_tr_fit = w_tr_eval if use_weights_for_training else None
    w_te_fit = w_te_eval if use_weights_for_training else None

    class_vals = np.array([0.0, 1.0, 2.0], dtype=float)

    def feval_weighted_qwk(preds: np.ndarray, dataset: lgb.Dataset):
        y_true = dataset.get_label().astype(int)
        proba = preds.reshape(len(y_true), 3, order="F")
        score = proba @ class_vals

        # Optimize thresholds on this eval set (quick grid) to approximate best QWK
        t1, t2, qwk = optimize_thresholds_for_weighted_qwk(
            y_true=y_true,
            score=score,
            sample_weight=dataset.get_weight(),
            n_grid=40
        )
        return ("weighted_qwk", float(qwk), True)

    params = {
        "objective": "multiclass",
        "num_class": 3,
        "metric": "None",  # custom only
        "boosting_type": "gbdt",
        "learning_rate": 0.05,
        "num_leaves": 63,
        "max_depth": -1,
        "min_data_in_leaf": 30,
        "feature_fraction": 0.9,
        "bagging_fraction": 0.9,
        "bagging_freq": 1,
        "lambda_l2": 1.0,
        "lambda_l1": 0.0,
        "verbosity": -1,
        "seed": split_seed,
        "feature_pre_filter": False,
        "force_row_wise": True,
    }

    dtrain = lgb.Dataset(X_tr.values, label=y_tr, weight=w_tr_fit, feature_name=feature_cols, free_raw_data=True)
    dvalid = lgb.Dataset(X_te.values, label=y_te, weight=w_te_fit, feature_name=feature_cols, reference=dtrain, free_raw_data=True)

    booster = lgb.train(
        params=params,
        train_set=dtrain,
        num_boost_round=num_boost_round,
        valid_sets=[dvalid],
        valid_names=["valid"],
        feval=feval_weighted_qwk,
        callbacks=[
            lgb.early_stopping(early_stopping_rounds, verbose=False),
            lgb.log_evaluation(period=0),
        ],
    )
    return booster

def rank_snps_by_shap_mace_lgbm(
    train: pd.DataFrame,
    target_col: str,
    base_cols: list[str],
    snp_cols: list[str],
    mace_label_weights: dict[int, float] = {0: 1.0, 1: 3.0, 2: 4.0},
    use_weights_for_training: bool = True,
    use_weights_for_shap: bool = True,
    test_size: float = 0.2,
    split_seed: int = 42,
    num_boost_round: int = 4000,
    early_stopping_rounds: int = 120,
) -> pd.DataFrame:

    snp_cols = [c for c in snp_cols if c in train.columns]
    if not snp_cols:
        raise ValueError("No SNP columns found in train.")

    feature_cols = base_cols + snp_cols
    y_all = train[target_col].astype(int)

    tr_idx, te_idx = train_test_split(
        train.index, test_size=test_size, stratify=y_all, random_state=split_seed
    )

    X_tr = train.loc[tr_idx, feature_cols].copy()
    X_te = train.loc[te_idx, feature_cols].copy()
    y_tr = y_all.loc[tr_idx].values
    y_te = y_all.loc[te_idx].values

    booster = _fit_lgbm_for_shap_multiclass(
        X_tr=X_tr,
        y_tr=y_tr,
        X_te=X_te,
        y_te=y_te,
        feature_cols=feature_cols,
        label_weights=mace_label_weights,
        use_weights_for_training=use_weights_for_training,
        split_seed=split_seed,
        num_boost_round=num_boost_round,
        early_stopping_rounds=early_stopping_rounds,
    )

    w_tr_eval = np.array([mace_label_weights[int(y)] for y in y_tr], dtype=float)
    shap_scores = _compute_shap_importance(
        booster,
        X_tr,
        feature_cols,
        use_weights_for_shap=use_weights_for_shap,
        sample_weight=w_tr_eval,
    )

    snp_scores = shap_scores.loc[snp_cols]
    rank_df = (
        snp_scores
        .rename("mean_abs_shap")
        .sort_values(ascending=False)
        .reset_index()
        .rename(columns={"index": "snp"})
    )
    rank_df["rank"] = np.arange(1, len(rank_df) + 1)

    return rank_df


# OUTCOME SEVERITY: baseline age, sex, genetics

In [59]:
fixed_cols_severity = ["Age_Baseline", "Genre"]

X = train[fixed_cols_severity + snp_cols].copy()
y = train["OUTCOME SEVERITY"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [60]:
def fit_lgbm_severity_rescaled_logloss_cv(
    train: pd.DataFrame,
    target_col: str,
    base_cols: list[str],
    snp_cols: list[str],
    stability_df: pd.DataFrame | None = None,
    selected_snp_top_n: int = 20,
    candidate_pool_size: int | None = None,
    use_pca: bool = True,
    n_pcs: int = 10,
    add_snp_sum: bool = True,
    test_size: float = 0.2,
    split_seed: int = 42,
    severity_label_weights: dict[int, float] | None = None,
    use_sample_weights: bool = True,  # affects TRAINING ONLY
    n_splits_cv: int = 5,
    n_param_samples: int = 40,
    num_boost_round: int = 4000,
    early_stopping_rounds: int = 100,
    verbose_cv: bool = False,
):

    if severity_label_weights is None:
        severity_label_weights = {0: 1.5, 1: 1.0}

    y_all = train[target_col].astype(int)
    tr_idx, te_idx = train_test_split(
        train.index, test_size=test_size, stratify=y_all, random_state=split_seed
    )
    y_tr = y_all.loc[tr_idx].values
    y_te = y_all.loc[te_idx].values

    # SNP ranking 
    snp_cols = [c for c in snp_cols if c in train.columns]
    if not snp_cols:
        raise ValueError("No SNP columns found in train.")

    if stability_df is not None and "snp" in stability_df.columns:
        ranked = [s for s in stability_df["snp"] if s in snp_cols]
        if not ranked:
            ranked = snp_cols.copy()
    else:
        ranked = snp_cols.copy()

    selected = ranked[:selected_snp_top_n]
    remaining = [s for s in ranked if s not in selected]
    candidate_pool = remaining if candidate_pool_size is None else remaining[:candidate_pool_size]

    if use_pca and len(candidate_pool) < max(n_pcs, 2):
        raise ValueError("Not enough SNPs in candidate_pool for PCA.")

    fixed_cols = base_cols + selected

    X_fixed_tr = train.loc[tr_idx, fixed_cols].copy()
    X_fixed_te = train.loc[te_idx, fixed_cols].copy()

    X_snp_tr = train.loc[tr_idx, candidate_pool].copy()
    X_snp_te = train.loc[te_idx, candidate_pool].copy()

    X_tr_df = X_fixed_tr.copy()
    X_te_df = X_fixed_te.copy()

    pca = None
    pc_cols: list[str] = []

    if use_pca:
        pca = PCA(n_components=n_pcs, random_state=split_seed)
        pcs_tr = pca.fit_transform(X_snp_tr.values)
        pcs_te = pca.transform(X_snp_te.values)
        pc_cols = [f"SNP_PC{i+1}" for i in range(n_pcs)]
        X_tr_df = pd.concat([X_tr_df, pd.DataFrame(pcs_tr, index=tr_idx, columns=pc_cols)], axis=1)
        X_te_df = pd.concat([X_te_df, pd.DataFrame(pcs_te, index=te_idx, columns=pc_cols)], axis=1)

    if add_snp_sum:
        # SNP sum is computed over the candidate_pool (matches original intent)
        X_tr_df["SNP_SUM"] = X_snp_tr.sum(axis=1).values
        X_te_df["SNP_SUM"] = X_snp_te.sum(axis=1).values

    feature_cols = list(X_tr_df.columns)


    # Evaluation weights (used for metrics)
    w_tr_eval = np.array([severity_label_weights[int(y)] for y in y_tr], dtype=float)
    w_te_eval = np.array([severity_label_weights[int(y)] for y in y_te], dtype=float)

    # Training weights
    w_tr_fit = w_tr_eval if use_sample_weights else None
    w_te_fit = w_te_eval if use_sample_weights else None

    # Baselines (computed on TRAIN distribution only; used to rescale fold scores)
    dummy_ll_tr = weighted_dummy_logloss_binary(y_tr, severity_label_weights)
    worst_ll_tr = weighted_worst_logloss_binary(y_tr, severity_label_weights)

    folds = make_stratified_folds(y_tr, n_splits=n_splits_cv, seed=split_seed)
    rng = np.random.default_rng(split_seed)

    best_score = -np.inf
    best_params: dict | None = None
    best_num_boost: int | None = None

    X_tr_np = X_tr_df.values

    for _ in range(n_param_samples):
        params = sample_params_binary(rng=rng)

        fold_scores: list[float] = []
        fold_best_iters: list[int] = []

        for fold in folds:
            tr_i, va_i = fold.train_idx, fold.valid_idx

            booster, it, p_va = train_one_fold_lgbm(
                params=params,
                X_tr=X_tr_np[tr_i],
                y_tr=y_tr[tr_i],
                X_va=X_tr_np[va_i],
                y_va=y_tr[va_i],
                feature_cols=feature_cols,
                num_boost_round=num_boost_round,
                early_stopping_rounds=early_stopping_rounds,
                verbose=verbose_cv,
                w_tr=None if w_tr_fit is None else w_tr_fit[tr_i],
                w_va=None if w_tr_fit is None else w_tr_fit[va_i],
            )

            # Fold metric: rescaled weighted logloss (higher is better)
            wll = weighted_logloss_binary(y_tr[va_i], p_va, severity_label_weights)
            rll = rescaled_weighted_logloss(wll, dummy_ll_tr, worst_ll_tr)

            fold_scores.append(float(rll))
            fold_best_iters.append(int(it))

        mean_rll = float(np.mean(fold_scores))
        mean_it = int(np.round(np.mean(fold_best_iters)))

        if mean_rll > best_score:
            best_score = mean_rll
            best_params = params
            best_num_boost = max(1, mean_it)

    if best_params is None or best_num_boost is None:
        raise RuntimeError("Severity CV search failed.")

# final train

    dtrain_full = lgb.Dataset(
        X_tr_df.values, label=y_tr, weight=w_tr_fit, feature_name=feature_cols, free_raw_data=True
    )
    dtest = lgb.Dataset(
        X_te_df.values, label=y_te, weight=w_te_fit, feature_name=feature_cols, reference=dtrain_full, free_raw_data=True
    )

    final_model = lgb.train(
        params=best_params,
        train_set=dtrain_full,
        num_boost_round=best_num_boost,
        valid_sets=[dtest],
        valid_names=["test"],
        callbacks=[
            lgb.early_stopping(early_stopping_rounds, verbose=False),
            lgb.log_evaluation(period=0),
        ],
    )
    best_it_final = int(final_model.best_iteration or best_num_boost)


    # Final evaluation

    p_tr = final_model.predict(X_tr_df.values, num_iteration=best_it_final)
    p_te = final_model.predict(X_te_df.values, num_iteration=best_it_final)

    wll_tr = weighted_logloss_binary(y_tr, p_tr, severity_label_weights)
    wll_te = weighted_logloss_binary(y_te, p_te, severity_label_weights)

    rll_tr = rescaled_weighted_logloss(wll_tr, dummy_ll_tr, worst_ll_tr)
    rll_te = rescaled_weighted_logloss(wll_te, dummy_ll_tr, worst_ll_tr)

    metrics = {
        "cv_best_rescaled_wll": float(best_score),
        "train_rescaled_wll": float(rll_tr),
        "test_rescaled_wll": float(rll_te),
        "best_num_boost_round": int(best_num_boost),
        "best_iteration_final": int(best_it_final),
        "used_sample_weights_for_training": bool(use_sample_weights),
    }

    return best_params, final_model, pca, metrics, candidate_pool, feature_cols, selected


In [61]:
sev_rank_df = rank_snps_by_shap_severity_lgbm(
    train=train,
    target_col="OUTCOME SEVERITY",
    base_cols=fixed_cols_severity,
    snp_cols=snp_cols,
    use_weights_for_training=True,
    use_weights_for_shap=True,
)

In [62]:
sev_rank_df

,snp,mean_abs_shap,rank
0,SNP21,0.119570,1
1,SNP282,0.082033,2
2,SNP10,0.070791,3
3,SNP99,0.051411,4
4,SNP175,0.039460,5
...,...,...,...
283,SNP28,0.000000,284
284,SNP38,0.000000,285
285,SNP30,0.000000,286
286,SNP31,0.000000,287


In [63]:
best_params_sev, model_sev, pca_sev, metrics_sev, pool_sev, feat_sev, trusted_sev = fit_lgbm_severity_rescaled_logloss_cv(
    train=train,
    target_col="OUTCOME SEVERITY",
    candidate_pool_size=80,
    base_cols=fixed_cols_severity,
    snp_cols=snp_cols,
    stability_df=sev_rank_df,
    severity_label_weights=severity_weights,
    use_sample_weights=True,
)
print(metrics_sev)

{'cv_best_rescaled_wll': 0.1313118020648695, 'train_rescaled_wll': 0.09415015860068765, 'test_rescaled_wll': 0.04662202644961766, 'best_num_boost_round': 200, 'best_iteration_final': 9, 'used_sample_weights_for_training': True}


In [64]:
#model_sev.save_model('severity_model_sub5.txt')

In [65]:
ID_COL = "trustii_id"
N_PCS = 10
pc_cols = [f"SNP_PC{i+1}" for i in range(N_PCS)]

# Base + trusted raw SNPs + remaining SNPs (for PCA/SNP_sum)
X_test = test[fixed_cols_severity + trusted_sev + pool_sev].copy()

# PCA transform 
if pca_sev is not None:
    pcs = pca_sev.transform(X_test[pool_sev].to_numpy())
    for i, c in enumerate(pc_cols):
        X_test[c] = pcs[:, i]

X_test["SNP_SUM"] = X_test[pool_sev].sum(axis=1)

X_test.drop(columns=pool_sev, inplace=True)
X_test = X_test[feat_sev]

proba_1 = model_sev.predict(
    X_test.values,
    num_iteration=getattr(model_sev, "best_iteration", None)
)

pred_df_severity = pd.DataFrame({
    ID_COL: test[ID_COL].values,
    "OUTCOME SEVERITY": proba_1, 
})
pred_df_severity


,trustii_id,OUTCOME SEVERITY
0,1,0.196824
1,2,0.290172
2,3,0.247374
3,4,0.236243
4,5,0.270367
...,...,...
144,145,0.266493
145,146,0.196824
146,147,0.303974
147,148,0.244773


# OUTCOME MACE: all baseline, at least 1-100 genes

In [66]:
fixed_cols_mace = [
    "Age_Baseline", "Age_Diag", "BMI", "BSA", "Genre",
    "Epaiss_max", "Gradient", "TVNS", "FEVG", "ATCD_MS", "SYNCOPE", "Diam_OG",
    "Variant.Pathogene"
]

X = train[fixed_cols_mace + snp_cols].copy()
y = train["OUTCOME MACE"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [67]:
def fit_lgbm_mace_qwk_cv(
    train: pd.DataFrame,
    target_col: str,
    base_cols: list[str],
    snp_cols: list[str],
    stability_df: pd.DataFrame | None = None,
    selected_snp_top_n: int = 20,
    candidate_pool_size: int | None = None,
    use_pca: bool = True,
    n_pcs: int = 10,
    add_snp_sum: bool = True,
    test_size: float = 0.2,
    split_seed: int = 42,
    mace_label_weights: dict[int, float] | None = None,
    use_sample_weights: bool = True,  # affects TRAINING ONLY
    n_splits_cv: int = 5,
    n_param_samples: int = 40,
    num_boost_round: int = 4000,
    early_stopping_rounds: int = 100,
    verbose_cv: bool = False,
    threshold_grid: int = 40,  # for threshold optimization inside CV
):

    if mace_label_weights is None:
        mace_label_weights = {0: 1.0, 1: 3.0, 2: 4.0}

    y_all = train[target_col].astype(int)
    tr_idx, te_idx = train_test_split(
        train.index, test_size=test_size, stratify=y_all, random_state=split_seed
    )
    y_tr = y_all.loc[tr_idx].values
    y_te = y_all.loc[te_idx].values

    classes = np.sort(np.unique(y_tr))
    if not np.array_equal(classes, np.array([0, 1, 2])):
        raise ValueError(f"{target_col} must have classes [0,1,2], got {classes.tolist()}")

    snp_cols = [c for c in snp_cols if c in train.columns]
    if not snp_cols:
        raise ValueError("No SNP columns found in train.")

    if stability_df is not None and "snp" in stability_df.columns:
        ranked = [s for s in stability_df["snp"] if s in snp_cols]
        if not ranked:
            ranked = snp_cols.copy()
    else:
        ranked = snp_cols.copy()

    selected = ranked[:selected_snp_top_n]
    remaining = [s for s in ranked if s not in selected]
    candidate_pool = remaining if candidate_pool_size is None else remaining[:candidate_pool_size]

    if use_pca and len(candidate_pool) < max(n_pcs, 2):
        raise ValueError("Not enough SNPs in candidate_pool for PCA.")
    
    fixed_cols = base_cols + selected

    X_fixed_tr = train.loc[tr_idx, fixed_cols].copy()
    X_fixed_te = train.loc[te_idx, fixed_cols].copy()

    X_snp_tr = train.loc[tr_idx, candidate_pool].copy()
    X_snp_te = train.loc[te_idx, candidate_pool].copy()

    X_tr_df = X_fixed_tr.copy()
    X_te_df = X_fixed_te.copy()

    pca = None
    pc_cols: list[str] = []

    if use_pca:
        pca = PCA(n_components=n_pcs, random_state=split_seed)
        pcs_tr = pca.fit_transform(X_snp_tr.values)
        pcs_te = pca.transform(X_snp_te.values)
        pc_cols = [f"SNP_PC{i+1}" for i in range(n_pcs)]
        X_tr_df = pd.concat([X_tr_df, pd.DataFrame(pcs_tr, index=tr_idx, columns=pc_cols)], axis=1)
        X_te_df = pd.concat([X_te_df, pd.DataFrame(pcs_te, index=te_idx, columns=pc_cols)], axis=1)

    if add_snp_sum:
        X_tr_df["SNP_SUM"] = X_snp_tr.sum(axis=1).values
        X_te_df["SNP_SUM"] = X_snp_te.sum(axis=1).values

    feature_cols = list(X_tr_df.columns)

    w_tr_eval = np.array([mace_label_weights[int(y)] for y in y_tr], dtype=float)
    w_te_eval = np.array([mace_label_weights[int(y)] for y in y_te], dtype=float)

    w_tr_fit = w_tr_eval if use_sample_weights else None
    w_te_fit = w_te_eval if use_sample_weights else None

    folds = make_stratified_folds(y_tr, n_splits=n_splits_cv, seed=split_seed)
    rng = np.random.default_rng(split_seed)

    best_qwk = -np.inf
    best_params: dict | None = None
    best_num_boost: int | None = None
    best_thresholds: tuple[float, float] | None = None

    X_tr_np = X_tr_df.values
    class_vals = np.array([0.0, 1.0, 2.0], dtype=float)

    for _ in range(n_param_samples):
        params = sample_params_multiclass(rng=rng, num_class=3)

        fold_scores: list[float] = []
        fold_best_iters: list[int] = []
        fold_t1s: list[float] = []
        fold_t2s: list[float] = []

        for fold in folds:
            tr_i, va_i = fold.train_idx, fold.valid_idx

            booster, it, proba_va = train_one_fold_lgbm(
                params=params,
                X_tr=X_tr_np[tr_i],
                y_tr=y_tr[tr_i],
                X_va=X_tr_np[va_i],
                y_va=y_tr[va_i],
                feature_cols=feature_cols,
                num_boost_round=num_boost_round,
                early_stopping_rounds=early_stopping_rounds,
                verbose=verbose_cv,
                w_tr=None if w_tr_fit is None else w_tr_fit[tr_i],
                w_va=None if w_tr_fit is None else w_tr_fit[va_i],
            )

            # Convert class-probabilities -> continuous ordinal score
            score_va = proba_va @ class_vals

            # Optimize thresholds on the validation fold to maximize weighted QWK
            t1, t2, qwk_va = optimize_thresholds_for_weighted_qwk(
                y_true=y_tr[va_i],
                score=score_va,
                sample_weight=w_tr_eval[va_i],
                n_grid=40,
            )

            fold_scores.append(float(qwk_va))
            fold_best_iters.append(int(it))
            fold_t1s.append(float(t1))
            fold_t2s.append(float(t2))

        mean_qwk = float(np.mean(fold_scores))
        mean_it = int(np.round(np.mean(fold_best_iters)))
        mean_t1 = float(np.mean(fold_t1s))
        mean_t2 = float(np.mean(fold_t2s))

        if mean_qwk > best_qwk:
            best_qwk = mean_qwk
            best_params = params
            best_num_boost = max(1, mean_it)
            best_thresholds = (mean_t1, mean_t2)

    if best_params is None or best_num_boost is None or best_thresholds is None:
        raise RuntimeError("MACE CV search failed.")

    dtrain_full = lgb.Dataset(
        X_tr_df.values, label=y_tr, weight=w_tr_fit, feature_name=feature_cols, free_raw_data=True
    )
    dtest = lgb.Dataset(
        X_te_df.values, label=y_te, weight=w_te_fit, feature_name=feature_cols, reference=dtrain_full, free_raw_data=True
    )

    final_model = lgb.train(
        params=best_params,
        train_set=dtrain_full,
        num_boost_round=best_num_boost,
        valid_sets=[dtest],
        valid_names=["test"],
        callbacks=[
            lgb.early_stopping(early_stopping_rounds, verbose=False),
            lgb.log_evaluation(period=0),
        ],
    )
    best_it_final = int(final_model.best_iteration or best_num_boost)

    proba_tr = final_model.predict(X_tr_df.values, num_iteration=best_it_final)
    proba_te = final_model.predict(X_te_df.values, num_iteration=best_it_final)

    score_tr = proba_tr @ class_vals
    score_te = proba_te @ class_vals

    t1, t2 = best_thresholds
    yhat_tr = apply_thresholds_ordinal(score_tr, t1, t2)
    yhat_te = apply_thresholds_ordinal(score_te, t1, t2)

    qwk_tr = quadratic_weighted_kappa(y_tr, yhat_tr, sample_weight=w_tr_eval)
    qwk_te = quadratic_weighted_kappa(y_te, yhat_te, sample_weight=w_te_eval)

    metrics = {
        "cv_best_qwk_weighted": float(best_qwk),
        "train_qwk_weighted": float(qwk_tr),
        "test_qwk_weighted": float(qwk_te),
        "best_threshold_t1": float(t1),
        "best_threshold_t2": float(t2),
        "best_num_boost_round": int(best_num_boost),
        "best_iteration_final": int(best_it_final),
        "used_sample_weights_for_training": bool(use_sample_weights),
    }

    return best_params, final_model, pca, metrics, candidate_pool, feature_cols, selected, best_thresholds


In [68]:
mace_rank_df = rank_snps_by_shap_mace_lgbm(
    train=train,
    target_col="OUTCOME MACE",
    base_cols=fixed_cols_mace,
    snp_cols=snp_cols,
    use_weights_for_training=True,
    use_weights_for_shap=True,
)

mace_rank_df

,snp,mean_abs_shap,rank
0,SNP278,0.201140,1
1,SNP117,0.095859,2
2,SNP137,0.084014,3
3,SNP259,0.064614,4
4,SNP205,0.062893,5
...,...,...,...
283,SNP230,0.000000,284
284,SNP228,0.000000,285
285,SNP233,0.000000,286
286,SNP229,0.000000,287


In [69]:
best_params_mace, model_mace, pca_mace, metrics_mace, pool_mace, feat_mace, trusted_mace, (t1, t2) = fit_lgbm_mace_qwk_cv(
    train=train,
    target_col="OUTCOME MACE",
    base_cols=fixed_cols_mace,
    snp_cols=snp_cols,
    stability_df=mace_rank_df,
    mace_label_weights=mace_weights,
    use_sample_weights=True,
)
print(metrics_mace)

{'cv_best_qwk_weighted': 0.45303757378010195, 'train_qwk_weighted': 0.6164918177402334, 'test_qwk_weighted': 0.29791140693137474, 'best_threshold_t1': 0.6926466558789389, 'best_threshold_t2': 0.9044883281153124, 'best_num_boost_round': 70, 'best_iteration_final': 40, 'used_sample_weights_for_training': True}


In [70]:
#model_mace.save_model('mace_model_sub5.txt')

In [71]:
ID_COL = "trustii_id"
N_PCS = 10
pc_cols = [f"SNP_PC{i+1}" for i in range(N_PCS)]

X_test = test[fixed_cols_mace + trusted_mace + pool_mace].copy()
pcs = pca_mace.transform(X_test[pool_mace].to_numpy())
for i, c in enumerate(pc_cols):
    X_test[c] = pcs[:, i]

X_test["SNP_SUM"] = X_test[pool_mace].sum(axis=1)
X_test.drop(columns=pool_mace, inplace=True)
X_test = X_test[feat_mace]

proba = model_mace.predict(
    X_test,
    num_iteration=getattr(model_mace, "best_iteration", None)  # uses best_iteration if present
)

if proba.ndim == 1:
    n_classes = 3  # for OUTCOME MACE {0,1,2}
    proba = proba.reshape(-1, n_classes)

# Convert probs -> predicted class {0,1,2}
pred_class = np.argmax(proba, axis=1).astype(int)

pred_df_mace = pd.DataFrame({
    ID_COL: test[ID_COL].values,
    "OUTCOME MACE": pred_class
})

pred_df_mace

,trustii_id,OUTCOME MACE
0,1,0
1,2,2
2,3,0
3,4,0
4,5,0
...,...,...
144,145,0
145,146,0
146,147,2
147,148,0


In [72]:
cardi5 = pd.merge(pred_df_mace, pred_df_severity, on='trustii_id')
#cardi5.to_csv('cardi5.csv', index=False)

In [73]:
cardi5

,trustii_id,OUTCOME MACE,OUTCOME SEVERITY
0,1,0,0.196824
1,2,2,0.290172
2,3,0,0.247374
3,4,0,0.236243
4,5,0,0.270367
...,...,...,...
144,145,0,0.266493
145,146,0,0.196824
146,147,2,0.303974
147,148,0,0.244773
